# WaveFront：minWM 的 WorldScore 子集评测（Colab）

做的事和 Delta 上的 `tools/eval/worldscore/run.sh` 完全一样，说明见同目录的 `README.md`。

- **GPU**：菜单「代码执行程序 → 更改运行时类型」选 **A100**（40 GB）。H100 也行（下面会给 DROID-SLAM 补上 sm_90）；L4 / T4 显存不够，第 2 格会直接报错。
- **输出**存在 Google Drive 的 `MyDrive/WaveFront/worldscore/<RUN_NAME>/`，完整子集约 6–7 GB（PNG 帧）。会话断了就重新从头运行全部单元格：环境要重装（约 30–40 分钟），已经生成和打完分的样例会跳过。
- **先冒烟**：默认 `EXTRA = "inference.limit=2"`，只生成并评 2 个样例，用来确认环境、测每个样例的耗时。确认没问题后把 `EXTRA` 改成 `""`、**RUN_NAME 不变**，再跑一遍，就接着冒烟的结果评完整子集。
- 仓库若是私有的，在左栏「密钥」里加一个 `GITHUB_TOKEN`（只需读权限），并允许本笔记本访问。

In [ ]:
# ---- 只改这里 ----
REPO_URL = "https://github.com/legallchaperone/WaveFront.git"
BRANCH = "eval/worldscore"
RUN_NAME = "2026-11-01-ws-minwm-s1"   # <日期>-<短名>；同一个名字 = 接着上次
INFER_CONFIG = "configs/wan21/action2v/infer/stage1_ar_tf.py"
CKPT_RELEASE, CKPT_LINK = "ar_diffusion_tf", "stage1_ar_tf"   # HF 上的名字 -> config 里的名字
EXTRA = "inference.limit=2"            # 冒烟：2 个样例；完整子集改成 ""
WS_PREPARE_ARGS = ""                   # 例如 "--static-per-group 2 --dynamic-per-group 2"（改了就换 RUN_NAME）
DRIVE_ROOT = "/content/drive/MyDrive/WaveFront/worldscore"
WS_COMMIT = "096c75fc4ada9c7d92e03140c3c1f0c8f383b61f"

import os
os.environ.update(
    BRANCH=BRANCH, INFER_CONFIG=INFER_CONFIG, CKPT_RELEASE=CKPT_RELEASE, CKPT_LINK=CKPT_LINK,
    EXTRA=EXTRA, WS_PREPARE_ARGS=WS_PREPARE_ARGS, WS_COMMIT=WS_COMMIT,
    OUT_DIR=f"{DRIVE_ROOT}/{RUN_NAME}",
    WS_ROOT="/content/WorldScore", WS_DATASET="/content/data/WorldScore-Dataset",
    WS_PYTHON="/content/venvs/ws/bin/python", UV_PYTHON_INSTALL_DIR="/content/uvpython",
)

In [ ]:
import subprocess
from google.colab import drive

drive.mount("/content/drive")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
mem = int(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"], text=True
).split()[0])
assert mem >= 38000, f"这张卡只有 {mem} MiB；minWM Stage 1 推理估计要 30+ GB，请换 A100 或 H100"
os.makedirs(os.environ["OUT_DIR"], exist_ok=True)

## 1. 代码与 minWM 环境（约 5 分钟）

In [ ]:
from google.colab import userdata

try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
if not os.path.isdir("/content/WaveFront/.git"):
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "/content/WaveFront"])
    assert r.returncode == 0, "clone 失败：分支推上去了吗？私有仓库要设 GITHUB_TOKEN"
else:
    subprocess.run(["git", "-C", "/content/WaveFront", "pull", "-q", "--ff-only"], check=True)
print(subprocess.check_output(["git", "-C", "/content/WaveFront", "log", "-1", "--oneline"], text=True))

In [ ]:
%%bash
set -eo pipefail
pip install -q uv
cd /content/WaveFront
[ -x /content/venvs/minwm/bin/python ] || uv venv -q --python 3.12 /content/venvs/minwm
uv pip install -q --python /content/venvs/minwm/bin/python -r requirements/base.txt
uv pip install -q --python /content/venvs/minwm/bin/python --no-deps -e .
/content/venvs/minwm/bin/python -c "import torch, minwm; print('torch', torch.__version__, 'cuda ok:', torch.cuda.is_available())"

In [ ]:
%%bash
# Wan 2.1 base（约 18 GB，其中 T5 编码器 11 GB）+ minWM 权重（6 GB）
set -eo pipefail
cd /content/WaveFront
export PATH=/content/venvs/minwm/bin:$PATH
hf download Wan-AI/Wan2.1-T2V-1.3B --local-dir ckpts/Wan2.1-T2V-1.3B \
    --include "Wan2.1_VAE.pth" "models_t5_umt5-xxl-enc-bf16.pth" "google/umt5-xxl/*" \
              "diffusion_pytorch_model.safetensors" "config.json" > /dev/null
hf download MIN-Lab/minWM --local-dir ckpts --include "Wan21/Action2V/$CKPT_RELEASE/*" > /dev/null
ln -sfnT "$CKPT_RELEASE" "ckpts/Wan21/Action2V/$CKPT_LINK"
du -shL ckpts/Wan2.1-T2V-1.3B "ckpts/Wan21/Action2V/$CKPT_LINK"

## 2. WorldScore 环境（约 30–40 分钟，主要是编译 DROID-SLAM）

按 WorldScore README 的 Evaluation 一节装，放在独立的 Python 3.10 环境里（它要 torch 2.5.1，minWM 用 2.9.1）。这一格最可能出问题，报错就把最后几十行贴回来。

In [ ]:
%%bash
set -eo pipefail
export CUDA_HOME=/usr/local/cuda PATH=/usr/local/cuda/bin:$PATH
export MAX_JOBS=$(nproc) AM_I_DOCKER=False BUILD_WITH_CUDA=True
cc=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1)
export TORCH_CUDA_ARCH_LIST=$cc
P=$WS_PYTHON
C=/content/ws-constraints.txt
printf 'torch==2.5.1\ntorchvision==0.20.1\ntorchaudio==2.5.1\nnumpy<2\nsetuptools<70\n' > $C

[ -d $WS_ROOT/.git ] || git clone -q https://github.com/haoyi-duan/WorldScore.git $WS_ROOT
cd $WS_ROOT
git checkout -q $WS_COMMIT
git submodule update -q --init --recursive thirdparty/DROID-SLAM
git submodule update -q --init thirdparty/Grounded-Segment-Anything thirdparty/sam2
apt-get -qq install -y libsuitesparse-dev > /dev/null

[ -x $P ] || uv venv -q --python 3.10 /content/venvs/ws
uv pip install -q --python $P -c $C torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
uv pip install -q --python $P -c $C xformers==0.0.28.post3 --index-url https://download.pytorch.org/whl/cu121
uv pip install -q --python $P -c $C torch-scatter -f https://data.pyg.org/whl/torch-2.5.1+cu121.html
uv pip install -q --python $P -c $C pip setuptools wheel ninja open3d tensorboard scipy opencv-python \
    tqdm matplotlib pyyaml evo gdown yacs loguru einops timm imageio spacy catalogue pyiqa \
    torchmetrics pytorch_lightning cvxpy

echo "== DROID-SLAM（编译较久）"
cd $WS_ROOT/thirdparty/DROID-SLAM
if [ "$cc" = "9.0" ]; then  # H100：它的 setup.py 只编到 sm_86
  grep -q compute_90 setup.py || sed -i "s/'-gencode=arch=compute_86,code=sm_86',/&\n                    '-gencode=arch=compute_90,code=sm_90',/" setup.py
fi
$P -c "import droid_backends, lietorch" 2>/dev/null || $P setup.py install > /content/droid_build.log 2>&1 \
  || { tail -40 /content/droid_build.log; exit 1; }

echo "== Grounded-SAM、SAM2、VFIMamba"
cd $WS_ROOT/thirdparty/Grounded-Segment-Anything
uv pip install -q --python $P -c $C -e segment_anything
uv pip install -q --python $P -c $C --no-build-isolation -e GroundingDINO
cd $WS_ROOT/thirdparty/sam2
uv pip install -q --python $P -c $C --no-build-isolation -e .
uv pip install -q --python $P -c $C --no-build-isolation causal_conv1d==1.5.0.post8 mamba_ssm==2.2.4

echo "== WorldScore"
uv pip install -q --python $P -c $C $WS_ROOT
$P -m spacy download -q en_core_web_sm
$P -c "import torch, droid_backends, lietorch, groundingdino, sam2, mamba_ssm, worldscore; print('WorldScore env ok, cuda:', torch.cuda.is_available())"

In [ ]:
%%bash
# 指标用的权重（约 5 GB）和数据集（1.2 GB）
set -eo pipefail
K=$WS_ROOT/worldscore/benchmark/metrics/checkpoints
mkdir -p $K && cd $K
get() { [ -s "$(basename "$1")" ] || wget -q "$1"; }
get https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
get https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
get https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
get https://huggingface.co/facebook/sam2.1-hiera-base-plus/resolve/main/sam2.1_hiera_base_plus.pt
get https://huggingface.co/MCG-NJU/VFIMamba_ckpts/resolve/main/ckpt/VFIMamba.pkl
[ -d models ] || { get https://dl.dropboxusercontent.com/s/4j4z58wuv8o0mfz/models.zip && unzip -q models.zip; }
# DROID-SLAM 官方权重（Google Drive）
[ -s droid.pth ] || /content/venvs/ws/bin/gdown -q 1PpqVt1H4maBa_GbPJp4NwxRsd9jk-elh -O droid.pth
ls -la $K

if [ ! -d $WS_DATASET ]; then
  mkdir -p /content/data && cd /content/data
  wget -q https://huggingface.co/datasets/Howieeeee/WorldScore/resolve/main/WorldScore-Dataset.zip
  unzip -q WorldScore-Dataset.zip && rm WorldScore-Dataset.zip
fi
ls $WS_DATASET

## 3. 生成 + 打分 + 汇总

生成时每条打印一行 `wrote ...`；打分阶段的输出带 `[shard 0]` 前缀，同时存进 Drive 上的 `eval-shard0.log`。

In [ ]:
%%bash
set -eo pipefail
cd /content/WaveFront
export PATH=/content/venvs/minwm/bin:$PATH TOKENIZERS_PARALLELISM=false
# shellcheck disable=SC2086
bash tools/eval/worldscore/run.sh "$INFER_CONFIG" $EXTRA 2>&1 | tee -a "$OUT_DIR/colab.log"

## 4. 看结果

In [ ]:
from IPython.display import Markdown, display

display(Markdown(open(f"{os.environ['OUT_DIR']}/summary.md", encoding="utf-8").read()))

In [ ]:
# 每种相机运动挑一个视频看
import base64
import glob
import json

from IPython.display import HTML

shown, html = set(), []
for d in sorted(glob.glob(f"{os.environ['OUT_DIR']}/worldscore_output/*/*/*/**/videos", recursive=True)):
    inst = os.path.dirname(d)
    data = json.load(open(f"{inst}/image_data.json"))
    key = data["camera_path"][0] if data["visual_movement"] == "static" else data["motion_type"]
    if key in shown:
        continue
    shown.add(key)
    b64 = base64.b64encode(open(f"{d}/output.mp4", "rb").read()).decode()
    html.append(f"<div style='display:inline-block;margin:4px'><b>{key}</b> {os.path.basename(inst)}<br>"
                f"<video width=416 controls loop src='data:video/mp4;base64,{b64}'></video></div>")
display(HTML("".join(html)))